This notebook shows the first attempt of importing the flight delay data

## Import Packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

Data is taken from this page: https://transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FGJ&QO_fu146_anzr=b0-gvzr

With the following columns selected:
* Year
* Quarter
* Month
* DayofMonth
* DayOfWeek
* FlightDate
* Reporting_Airline
* Flight_Number_Reporting_Airline
* OriginAirportID
* Origin
* DestAirportID
* Dest
* ArrDelayMinutes
* ArrDel15

## Import Data 

In [2]:
df = pd.read_csv("../data/raw/flights_2026_01")
df.shape

(544003, 15)

In [3]:
df = df.sample(100_000)

In [4]:
df.shape

(100000, 15)

In [5]:
df.head()

,Unnamed: 0,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,ARR_DELAY_NEW,ARR_DEL15
14783,14783,2026,1,1,1,4,1/1/2026 12:00:00 AM,WN,2746,12191,HOU,12953,LGA,0.0,0.0
237248,237248,2026,1,1,14,3,1/14/2026 12:00:00 AM,DL,1477,10397,ATL,13930,ORD,78.0,1.0
83402,83402,2026,1,1,5,1,1/5/2026 12:00:00 AM,F9,1549,12451,JAX,10397,ATL,121.0,1.0
376425,376425,2026,1,1,22,4,1/22/2026 12:00:00 AM,DL,2610,13487,MSP,13486,MSO,29.0,1.0
80889,80889,2026,1,1,5,1,1/5/2026 12:00:00 AM,DL,1113,11433,DTW,11292,DEN,0.0,0.0


In [6]:
df.columns

Index(['Unnamed: 0', 'YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK',
       'FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM',
       'ORIGIN_AIRPORT_ID', 'ORIGIN', 'DEST_AIRPORT_ID', 'DEST',
       'ARR_DELAY_NEW', 'ARR_DEL15'],
      dtype='str')

In [7]:
df = df.drop("Unnamed: 0", axis=1) # drop index column

In [8]:
df.isna().sum()

YEAR                    0
QUARTER                 0
MONTH                   0
DAY_OF_MONTH            0
DAY_OF_WEEK             0
FL_DATE                 0
OP_UNIQUE_CARRIER       0
OP_CARRIER_FL_NUM       0
ORIGIN_AIRPORT_ID       0
ORIGIN                  0
DEST_AIRPORT_ID         0
DEST                    0
ARR_DELAY_NEW        4918
ARR_DEL15            4918
dtype: int64

* Only missing columns in this case are the dependant variable so we have to drop

In [9]:
df[['ARR_DELAY_NEW','ARR_DEL15']].dropna(inplace=True) # drop all missing dependent rows

In [10]:
df.dtypes

YEAR                   int64
QUARTER                int64
MONTH                  int64
DAY_OF_MONTH           int64
DAY_OF_WEEK            int64
FL_DATE                  str
OP_UNIQUE_CARRIER        str
OP_CARRIER_FL_NUM      int64
ORIGIN_AIRPORT_ID      int64
ORIGIN                   str
DEST_AIRPORT_ID        int64
DEST                     str
ARR_DELAY_NEW        float64
ARR_DEL15            float64
dtype: object

* Most datatypes make sense except FL_DATE should be datetime
* ARR_DEL15 should be binary

In [11]:
df['FL_DATE'].sample(10)

473811    1/27/2026 12:00:00 AM
492244    1/29/2026 12:00:00 AM
312773    1/18/2026 12:00:00 AM
432467    1/25/2026 12:00:00 AM
381197    1/22/2026 12:00:00 AM
82136      1/5/2026 12:00:00 AM
109271     1/6/2026 12:00:00 AM
115916     1/7/2026 12:00:00 AM
204428    1/12/2026 12:00:00 AM
530638    1/31/2026 12:00:00 AM
Name: FL_DATE, dtype: str

* FL_DATE does not contain scheduled departure time, we need to get the CRSDepTime column (which is not the same as actual deperature time DEPTime)

In [12]:
pd.to_datetime(df['FL_DATE'], format='mixed', errors='coerce')

14783    2026-01-01
237248   2026-01-14
83402    2026-01-05
376425   2026-01-22
80889    2026-01-05
            ...    
142697   2026-01-08
259702   2026-01-15
473632   2026-01-27
455768   2026-01-26
532421   2026-01-31
Name: FL_DATE, Length: 100000, dtype: datetime64[us]

* May not require FL_Date due to already have the other date information

In [13]:
df['OP_UNIQUE_CARRIER'].value_counts()

OP_UNIQUE_CARRIER
WN    19049
DL    14604
AA    14025
OO    12103
UA    11645
YX     5475
AS     4837
MQ     4383
OH     3616
B6     3434
F9     2906
NK     2194
G4     1729
Name: count, dtype: int64

* It would be nice to use a lookup table to easily understand which airline is which

In [14]:
df['OP_CARRIER_FL_NUM'].nunique()

6182

* There are many flight numbers which might not be helpful in predictions, perhaps they can be combined with the airline to identify troublesome routes, though the combination of airline, origin, destination might do that already
* Probably safe to drop this column for now

In [15]:
df['ORIGIN_AIRPORT_ID'].nunique(), df['ORIGIN'].nunique()

(341, 341)

These columns convey the same information so one of them is unnecessary, probably can drop the ORIGIN because it is recommended to use ORIGIN_AIRPORT_ID for cross year analysis

Same situation with DEST_AIRPORT_ID and DEST

In [17]:
df.columns

Index(['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE',
       'OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM', 'ORIGIN_AIRPORT_ID', 'ORIGIN',
       'DEST_AIRPORT_ID', 'DEST', 'ARR_DELAY_NEW', 'ARR_DEL15'],
      dtype='str')

In [20]:
df[['ARR_DELAY_NEW','ARR_DEL15']].sample(5)

,ARR_DELAY_NEW,ARR_DEL15
391678,44.0,1.0
3713,29.0,1.0
399405,0.0,0.0
138716,37.0,1.0
374908,0.0,0.0


* Have to decide between a classification task and a regression task
* Either predict if flight will land more than fifteen minutes late or predict exactly how many minutes delayed (if at all it will land)

In [20]:
df['route'] = df['ORIGIN'] + '-' + df['DEST']
len(df['route'].unique())

5505

* 5505 unique routes from 100_000 sample suggests a reasonable number of data for each route
* There may be some less run routes with limited data and therefore missed by sampling

## Final Thoughts

* This dataset is fairly simple and does not require much preprocessing
* Will need to reimport import the dataset with a better choice of columns, it would be good to use a lookup table to convert all the Airports and Carriers into a recognizable format
* Will need to make a pipeline which says what to do with missing variables considering how the user will input the flight
* Stil missing some important information related to flight delays - weather
* Need to find a simple heuristic to use as a benchline, e.g. propoprtion of all flights from airline on a particular route which are delayed
* Will need a full year of data to account for seasonal effects
  * Can sample 20_000 - 30_000 samples from each month